In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it-qat-q4_0-unquantized/2/generation_config.json
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competit

In [7]:
!pip install gensim wandb wikipedia-api langchain langchain_text_splitters langchain-community langchain-huggingface faiss-cpu transformers accelerate bitsandbytes --quiet
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not 

In [3]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import wandb
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [4]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
 
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:\n", train_df.isnull().sum())
print("\nAnswer label distribution:\n", train_df["answer"].value_counts())
 
train_df["prompt_len"] = train_df["prompt"].astype(str).apply(lambda x: len(x.split()))
print("\nPrompt word-length stats:\n", train_df["prompt_len"].describe())

print(train_df.head(3))

Train shape: (2000, 8)
Test shape : (500, 7)

Missing values in train:
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer label distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt word-length stats:
 count    2000.00000
mean       18.14650
std         6.78189
min         3.00000
25%        14.00000
50%        17.00000
75%        22.00000
max        51.00000
Name: prompt_len, dtype: float64
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   

                         

In [14]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
 
TEXT_COLS = ["prompt", "A", "B", "C", "D", "E"]
 
for col in TEXT_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

In [15]:
def tokenize(text):
    return text.split()

In [16]:
all_sentences = []
for df in [train_df, test_df]:
    for col in TEXT_COLS:
        all_sentences.extend(df[col].apply(tokenize).tolist())
 
EMBED_DIM = 100
 
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=SEED,
)
 
print("Vocabulary size:", len(w2v_model.wv))
w2v_model.save(os.path.join("/kaggle/working", "word2vec.model"))

Vocabulary size: 2973


In [17]:
def text_to_vector(text, model, dim=EMBED_DIM):
    words = tokenize(text)
    vecs = []

    for word in words:
        if word in model.wv:
            vecs.append(model.wv[word])

    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)

    avg_vector = np.mean(vecs, axis=0)
    return avg_vector.astype(np.float32)

In [18]:
LABELS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

class MCQDataset(Dataset):

    def __init__(self, df, w2v_model, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.model = w2v_model
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        question_vector = text_to_vector(row["prompt"], self.model)

        features = []

        for option in LABELS:
            option_vector = text_to_vector(row[option], self.model)

            difference = np.abs(question_vector - option_vector)

            feature = np.concatenate((question_vector, option_vector, difference))

            features.append(feature)

        features = np.array(features)

        data = {}
        data["features"] = torch.tensor(features, dtype=torch.float32)

        if self.has_labels:
            answer = LABEL2IDX[row["answer"]]
            data["label"] = torch.tensor(answer, dtype=torch.long)
        else:
            data["id"] = row["id"]

        return data

In [19]:
class MCQScorer(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
 
    def forward(self, x):
        batch, n_options, dim = x.shape
        x = x.view(batch * n_options, dim)
        scores = self.net(x)
        scores = scores.view(batch, n_options)
        return scores

In [21]:
def map_at_3(probs, labels):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = []

    for pred, true in zip(top3, labels):
        if true in pred:
            score.append(1 / (np.where(pred == true)[0][0] + 1))
        else:
            score.append(0)

    return np.mean(score)


def run_epoch(model, loader, optimizer, criterion, train=True):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    for batch in loader:

        x = batch["features"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        with torch.set_grad_enabled(train):

            output = model(x)
            loss = criterion(output, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)

        all_probs.append(torch.softmax(output, dim=1).cpu().detach().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    loss = total_loss / len(loader.dataset)
    acc = (all_probs.argmax(1) == all_labels).mean()
    map3 = map_at_3(all_probs, all_labels)

    return loss, acc, map3


try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except:
    pass

wandb.login()

wandb.init(
    project="dlgenai-project-26t2",
    config={
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "hidden_dim": 128
    }
)

cfg = wandb.config

train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["answer"]
)

train_loader = DataLoader(
    MCQDataset(train_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    MCQDataset(val_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=False
)

model = MCQScorer(3 * EMBED_DIM, cfg.hidden_dim).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()

best_map = 0

for epoch in range(cfg.epochs):

    train_loss, train_acc, train_map = run_epoch(
        model, train_loader, optimizer, criterion, True
    )

    val_loss, val_acc, val_map = run_epoch(
        model, val_loader, optimizer, criterion, False
    )

    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_map3": train_map,
        "val_map3": val_map
    })

    print(f"Epoch {epoch+1}  Validation MAP@3 = {val_map:.4f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), "/kaggle/working" + "/best_model.pt")

wandb.finish()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


train_loss,▁
train_map3,▁
val_loss,▁
val_map3,▁
train_loss,1.58099
train_map3,0.47402
val_loss,1.49941
val_map3,0.62833


Epoch 1  Validation MAP@3 = 0.6450
Epoch 2  Validation MAP@3 = 0.6767
Epoch 3  Validation MAP@3 = 0.6961
Epoch 4  Validation MAP@3 = 0.7344
Epoch 5  Validation MAP@3 = 0.7494
Epoch 6  Validation MAP@3 = 0.7811
Epoch 7  Validation MAP@3 = 0.8167
Epoch 8  Validation MAP@3 = 0.8161
Epoch 9  Validation MAP@3 = 0.8450
Epoch 10  Validation MAP@3 = 0.8606
Epoch 11  Validation MAP@3 = 0.8444
Epoch 12  Validation MAP@3 = 0.8867
Epoch 13  Validation MAP@3 = 0.8711
Epoch 14  Validation MAP@3 = 0.8883
Epoch 15  Validation MAP@3 = 0.8989
Epoch 16  Validation MAP@3 = 0.9067
Epoch 17  Validation MAP@3 = 0.8933
Epoch 18  Validation MAP@3 = 0.9133
Epoch 19  Validation MAP@3 = 0.9233
Epoch 20  Validation MAP@3 = 0.9267


train_loss,█▇▅▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁
train_map3,▁▃▄▅▅▆▆▇▆▇▇▇▇▇██████
val_loss,█▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val_map3,▁▂▂▃▄▄▅▅▆▆▆▇▇▇▇█▇███
train_loss,0.58649
train_map3,0.84686
val_loss,0.41423
val_map3,0.92667


In [23]:
model = MCQScorer(3 * EMBED_DIM, 128).to(DEVICE)
model.load_state_dict(torch.load("/kaggle/working" + "/best_model.pt", map_location=DEVICE))
model.eval()

test_loader = DataLoader(
    MCQDataset(test_df, w2v_model, has_labels=False),
    batch_size=32,
    shuffle=False
)

ids = []
predictions = []

with torch.no_grad():

    for batch in test_loader:

        x = batch["features"].to(DEVICE)

        probs = torch.softmax(model(x), dim=1).cpu().numpy()
        top3 = np.argsort(-probs, axis=1)[:, :3]

        for i in range(len(top3)):
            ids.append(int(batch["id"][i]))
            predictions.append(" ".join(LABELS[j] for j in top3[i]))

test_df["Prediction_Model_NN"] = predictions
print("NN Model predictions saved...")

NN Model predictions saved...


In [5]:
wiki_topics = [
    "Supersymmetric quantum mechanics", "Heisenberg uncertainty principle", "Virtual particle",
    "Spontaneous symmetry breaking", "Wigner distribution function", "Magnetic monopole",
    "Spin quantum number", "Parity (physics)", "Peierls bracket", "Geometric quantization",
    "Quantum field theory", "Hilbert space", "Probability amplitude", "Ramsauer–Townsend effect",
    "Explicit symmetry breaking", "Angular momentum operator", "Standard Model",
    "Higgs boson", "CP violation", "Quark", "Chemical potential",
    "Lorentz covariance", "Minkowski space", "Minkowski diagram", "Special relativity",
    "General relativity", "Simultaneity", "Speed of light", "Born reciprocity",
    "Frame-dragging", "Gravitomagnetism", "Gravity Probe B", "Roche limit",
    "Penrose process", "Black hole information paradox", "Schwarzschild black hole",
    "CEERS-93316", "James Webb Space Telescope", "Redshift", "Metric expansion of space",
    "Proper distance", "Interstellar medium", "Molecular cloud", "Supernova remnant",
    "Main sequence", "Pulsar", "Crab Pulsar", "Supermassive black hole",
    "Sagittarius A*", "Dark matter", "Gravitational wave", "Doppler effect",
    "Lyman-alpha line", "Planetary system", "X-ray pulsar-based navigation",
    "Baryon acoustic oscillations", "Modified Newtonian dynamics", "Inflaton",
    "Einstein@Home", "Light-year", "Apparent magnitude", "Metallicity",
    "Kapteyn's Star", "Isophote", "Type Ia supernova", "Supernova",
    "Carnot heat engine", "Maxwell's demon", "Throttling process", "Second law of thermodynamics",
    "Kelvin–Helmholtz instability", "Coherent turbulent structure", "Cavitation", "Convection",
    "Natural convection", "Bernoulli's principle", "Kutta condition", "Navier–Stokes equations",
    "Cauchy momentum equation", "Water hammer",
    "Fermat's principle", "Emissivity", "Illuminance", "Luminance", "Rayleigh scattering",
    "Young's interference experiment", "Diffraction", "Total internal reflection",
    "Radiosity (radiometry)", "Stefan–Boltzmann law", "Ultraviolet catastrophe",
    "Optical signal-to-noise ratio", "Propagation constant", "Loudness",
    "Landau–Lifshitz–Gilbert equation", "Magnetic susceptibility", "Memristor",
    "Spin valve", "Electrical resistivity and conductivity", "Superconductivity",
    "Amorphous metal", "Variable-range hopping", "Piezoelectricity", "Dielectric loss",
    "Josephson effect", "De Haas–Van Alphen effect", "Paramagnetism", "Order parameter",
    "Ferroelectricity", "ReRAM", "Evans balance", "Spatial dispersion",
    "Identity element", "Crystallographic point group", "Improper rotation",
    "Crystallinity", "API gravity", "Radiometric dating", "Recrystallization (metallurgy)", "Grain boundary strengthening", 
    "Fischer–Tropsch process", "Carbocation", "Naphthalene", "Crossover experiment",
    "Fourier-transform infrared spectroscopy", "Three moment theorem", "Bollard pull", "Ring-imaging Cherenkov detector", 
    "Formal system", "Uniform tilings in hyperbolic plane", "Regular polytope",
    "Probability density function", "Probability mass function", "Reciprocal length",
    "Symmetry group", "Erlangen program", "Hyperbolic geometry", "Permutation group",
    "CW complex", "Dimension", "Hesse's principle of transfer", "Dynamic scaling", "Liouville's theorem (Hamiltonian)", 
    "Surgical pathology", "Active transport", "Trophic level", "Pulmonary circulation",
    "Mammary gland", "Organography", "Cyclotide", "Phageome",
    "Myrmecophyte", "Mycorrhiza", "IL-10", "Regulatory T cell", "Anatomy", "Cardiac skeleton",
    "Second", "Coordinated Universal Time", "Universal Time",
    "Triskelion", "Newton's laws of motion", "Right-hand rule", "Giordano Bruno",
    "Shower-curtain effect", "Wilson cloud chamber", "Ozma Problem", "Horror vacui",
    "Butterfly effect", "Gauss's law", "Scale (map)", "Martin Heidegger",
    "Isaac Newton", "Robert Hooke", "Pierre de Fermat",
    "Classical mechanics", "Earnshaw's theorem", "Environmental Science Center",
    "Memristor", "Synaptic transistor", "Power density", "Cold dark matter", "Antimatter",
    "Baryon asymmetry", "L dwarf",
    "Pycnometer", "Photophoresis", "Isophote", "Recycling", "Rare-earth element",
    "Fusor", "Thylakoid", "Diquark", "Thermodynamic system", "Molecular symmetry",
    "Mass-to-charge ratio", "Rømer's determination of the speed of light",
    "Resistive random-access memory", "Diffuse sky radiation", "Grain growth",
]

In [8]:
import os
import wikipediaapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

wiki = wikipediaapi.Wikipedia(
    user_agent="MyRAGProject/1.0 (singhshikhar8957@gmail.com)",
    language="en"
)

print("Scraping Wikipedia to build the Knowledge Base...")

scraped_texts = []

for topic in wiki_topics:
    try:
        page = wiki.page(topic)
        if page.exists():
            scraped_texts.append(page.text)
        else:
            print(f"Skipped {topic}: page not found")
    except Exception as e:
        print(f"Skipped {topic} due to error: {e}")
print(f"Successfully scraped {len(scraped_texts)} articles.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=300)
docs = text_splitter.create_documents(scraped_texts)
print(f"Created {len(docs)} chunks.")

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", 
                                   model_kwargs={"device": DEVICE,
                                                "model_kwargs": {"use_safetensors": False}},
                                   encode_kwargs={"normalize_embeddings": True}
)
vector_db = FAISS.from_documents(docs, embeddings)

FAISS_SAVE_PATH = "/kaggle/working/faiss_index"
vector_db.save_local(FAISS_SAVE_PATH)
print(f"FAISS index created successfully and saved to {FAISS_SAVE_PATH}")

/tmp/ipykernel_58/2862131934.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Scraping Wikipedia to build the Knowledge Base...
Skipped Maximal acceleration (physics): page not found
Skipped Cyclidae: page not found
Skipped CYCLOIDEA: page not found
Skipped Environmental Science Center, Qatar University: page not found
Skipped Atomristor: page not found
Skipped Synapstor: page not found
Skipped Class L dwarf: page not found
Successfully scraped 194 articles.


Created 11263 chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created successfully and saved to /kaggle/working/faiss_index


In [1]:
import transformers
from transformers import AutoTokenizer, AutoModelForMultimodalLM, pipeline
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch

transformers.logging.set_verbosity_error()

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(hf_token)
    print("Logged into Hugging Face successfully!")
except Exception as e:
    print(f"HF login failed: {e}. Please ensure you have added a 'HF_TOKEN' secret in Kaggle.")

model_id = "google/gemma-4-12B-it-qat-q4_0-unquantized" 

print(f"Loading {model_id} from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

llm_model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

llm_pipe = pipeline(
    "text-generation", 
    model=llm_model, 
    tokenizer=tokenizer, 
    max_new_tokens=20, 
    do_sample=True, 
    temperature=0.5,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)
print("LLM loaded successfully and ready for inference!")

Logged into Hugging Face successfully!
Loading google/gemma-4-12B-it-qat-q4_0-unquantized from Hugging Face...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

ValueError: The checkpoint you are trying to load has model type `gemma4_unified` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading offline FAISS Vector Database...")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5", 
    model_kwargs={"device": DEVICE}
)

FAISS_SAVE_PATH = "/kaggle/working/faiss_index"

loaded_vector_db = FAISS.load_local(
    FAISS_SAVE_PATH, 
    embeddings, 
    allow_dangerous_deserialization=True
)
print("Knowledge Base loaded successfully!")

/tmp/ipykernel_206/1746756083.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading offline FAISS Vector Database...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge Base loaded successfully!


In [1]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

if "Prediction_Model_NN" not in test_df:
    test_df["Prediction_Model_NN"] = "A B C"

wandb.init(
    project="dlgenai-project-26t2", 
    job_type="rag_inference", 
    name="gemma-4-12b-rag")

rag_table = wandb.Table(columns=[
    "ID", "Question", "Retrieved_Context", "Raw_LLM_Output", 
    "Final_Prediction", "Used_Fallback"])

def predict_with_rag(row):
    prompt_text = str(row['prompt'])
    fallback = row['Prediction_Model_NN']
    
    search_results = loaded_vector_db.similarity_search(prompt_text, k=5)
    retrieved_context = " ".join([doc.page_content for doc in search_results])
    
    full_prompt = f"""You are an expert scientist taking a multiple-choice test. 
Read the context carefully and rank the top 3 most likely correct options. 

Context: {retrieved_context}

Question: {prompt_text}
A: {row['A']}
B: {row['B']}
C: {row['C']}
D: {row['D']}
E: {row['E']}

Task: Output ONLY the 3 best letters separated by a space (Example: A B C). Do not write any explanations.
Answer:"""
    
    raw_output = ""
    final_ans_str = fallback
    used_fallback = True
    
    try:
        raw_output = llm_pipe(full_prompt)[0]['generated_text']
        llm_preds = []
        for char in raw_output.upper():
            if char in "ABCDE" and char not in llm_preds:
                llm_preds.append(char)
                
        fallback_preds = fallback.split()
        
        for fb_char in fallback_preds:
            if len(llm_preds) < 3 and fb_char not in llm_preds:
                llm_preds.append(fb_char)
                used_fallback = True
        final_ans_str = " ".join(llm_preds[:3])
            
    except Exception as e:
        raw_output = f"ERROR: {str(e)}"
        final_ans_str = fallback
        used_fallback = True
        
    return final_ans_str

print("Starting RAG Inference.....")
test_df['Final_Prediction'] = test_df.apply(predict_with_rag, axis=1)
print("Inference Complete!")

wandb.log({"rag_evaluation_results": rag_table})
wandb.finish()

final_submission = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": test_df["Final_Prediction"]
})

final_submission.to_csv("submission.csv", index=False)
print(final_submission.head())

NameError: name 'pd' is not defined

In [ ]:
final_submission = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": test_df["Final_Prediction"]})

final_submission.to_csv("submission.csv", index=False)
print(final_submission.head())